# M6 — Multi-Criterion Composite Ranking (Manuscript, Supervisor Feedback Item A-3)

**Purpose.** The paper already discloses that the validation-set ranking of the 30 marathon runs
disagrees with their hospital-level ranking (Spearman rho = 0.271, from `M2_consensus_selection.ipynb`).
Item A-3 asks whether a composite ranking across validation AUROC, hospital-level AUROC, and
hospital-level ECE would have picked a more robust expression than the validation-only criterion did.

**Plan.**
1. Load `M2_consensus_ranking.csv` (already computed: validation AUROC + rank, hospital-level mean
   AUROC + rank, hospital-level ECE, for all 30 marathon runs — no new experiments).
2. Add a rank for hospital-level ECE (ascending — lower ECE is better).
3. Compute a composite rank per run as the mean of the three individual ranks (validation AUROC rank,
   hospital AUROC rank, hospital ECE rank).
4. Sort by composite rank; identify the composite winner and compare it to the canonical run (seed 14).
5. Save the combined table and report the comparison.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT = Path(r"C:\ML PROJECT\sepsis-gp")
TABLES_SRC = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_SEED = 14

m2 = pd.read_csv(TABLES_SRC / "M2_consensus_ranking.csv")
print(f"Loaded {len(m2)} runs from M2_consensus_ranking.csv")
m2.head()

Loaded 30 runs from M2_consensus_ranking.csv


,run,is_canonical,complexity_used,val_auroc,rank_val,loho_auroc_mean,rank_loho,loho_ece_overall,n_hospitals_evaluable
0,14,True,24,0.7640,1,0.730058,19,0.008896,65
1,25,False,23,0.7592,2,0.730415,17,0.010445,65
2,13,False,24,0.7585,3,0.737822,2,0.012397,65
3,12,False,24,0.7569,4,0.733287,7,0.003364,65
4,11,False,24,0.7563,5,0.731072,16,0.015070,65


## Step 1 — Add hospital-level ECE rank, compute composite rank

In [2]:
m2["rank_ece"] = m2["loho_ece_overall"].rank(ascending=True, method="min").astype(int)
m2["composite_rank_value"] = m2[["rank_val", "rank_loho", "rank_ece"]].mean(axis=1)
m2["composite_rank"] = m2["composite_rank_value"].rank(ascending=True, method="min").astype(int)

composite_sorted = m2.sort_values("composite_rank_value").reset_index(drop=True)
cols = ["run", "is_canonical", "val_auroc", "rank_val", "loho_auroc_mean", "rank_loho",
        "loho_ece_overall", "rank_ece", "composite_rank_value", "composite_rank"]
composite_sorted[cols]

,run,is_canonical,val_auroc,rank_val,loho_auroc_mean,rank_loho,loho_ece_overall,rank_ece,composite_rank_value,composite_rank
0,12,False,0.7569,4,0.733287,7,0.003364,2,4.333333,1
1,8,False,0.7537,13,0.732307,10,0.003839,4,9.000000,2
2,22,False,0.7500,22,0.735634,4,0.002132,1,9.000000,2
3,7,False,0.7547,9,0.732182,11,0.007564,13,11.000000,4
4,26,False,0.7533,15,0.737161,3,0.008862,15,11.000000,4
5,13,False,0.7585,3,0.737822,2,0.012397,28,11.000000,4
6,14,True,0.7640,1,0.730058,19,0.008896,16,12.000000,7
7,6,False,0.7541,11,0.731692,13,0.007551,12,12.000000,7
8,23,False,0.7538,12,0.731289,15,0.006119,9,12.000000,7
9,28,False,0.7529,16,0.731683,14,0.005987,7,12.333333,10


## Step 2 — Composite winner vs. canonical run 14

In [3]:
winner = composite_sorted.iloc[0]
canonical_row = m2[m2["run"] == CANONICAL_SEED].iloc[0]

print("=== Composite ranking winner ===")
print(f"Run {int(winner['run'])}: val_auroc={winner['val_auroc']:.4f} (rank {int(winner['rank_val'])}), "
      f"hosp_auroc={winner['loho_auroc_mean']:.4f} (rank {int(winner['rank_loho'])}), "
      f"hosp_ece={winner['loho_ece_overall']:.4f} (rank {int(winner['rank_ece'])}), "
      f"composite_rank={int(winner['composite_rank'])}")

print(f"\n=== Canonical run {CANONICAL_SEED} under composite ranking ===")
print(f"val_auroc={canonical_row['val_auroc']:.4f} (rank {int(canonical_row['rank_val'])}), "
      f"hosp_auroc={canonical_row['loho_auroc_mean']:.4f} (rank {int(canonical_row['rank_loho'])}), "
      f"hosp_ece={canonical_row['loho_ece_overall']:.4f} (rank {int(canonical_row['rank_ece'])}), "
      f"composite_rank={int(canonical_row['composite_rank'])}")

print(f"\nIs the composite winner the same as the canonical run? {int(winner['run']) == CANONICAL_SEED}")

=== Composite ranking winner ===
Run 12: val_auroc=0.7569 (rank 4), hosp_auroc=0.7333 (rank 7), hosp_ece=0.0034 (rank 2), composite_rank=1

=== Canonical run 14 under composite ranking ===
val_auroc=0.7640 (rank 1), hosp_auroc=0.7301 (rank 19), hosp_ece=0.0089 (rank 16), composite_rank=7

Is the composite winner the same as the canonical run? False


In [4]:
out_path = OUT_DIR / "M6_composite_ranking.csv"
composite_sorted[cols].to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: C:\ML PROJECT\sepsis-gp\results\manuscript\tables\M6_composite_ranking.csv


## Findings

The composite ranking, computed as the mean of three individual ranks (validation AUROC, hospital-level
mean AUROC, hospital-level ECE) across the 30 marathon runs, selected run 12 as the top-ranked candidate
(composite rank 1 of 30; mean rank 4.33). Run 12 ranked 4th on validation AUROC (0.7569), 7th on
hospital-level AUROC (0.7333), and 2nd on hospital-level ECE (0.0034) — a consistently strong candidate
across all three criteria, with no single weak dimension.

The canonical expression (run 14), which ranked 1st under the validation-only criterion used for formula
selection (Equation 3), fell to 7th under the composite criterion (mean rank 12.0). Its hospital-level
AUROC rank (19th of 30, 0.7301) and hospital-level ECE rank (16th of 30, 0.0089) were markedly weaker than
its validation rank, consistent with the validation-vs-hospital rank disagreement already reported
(Spearman rho = 0.271, `M2_consensus_selection.ipynb`).

This confirms that multi-criterion selection incorporating both patient-level and hospital-level
performance would have identified a different, more consistently-ranked candidate than validation-AUROC-only
selection did. The result does not by itself prove the composite winner generalises better to a genuinely
unseen hospital — both criteria are computed on data available at formula-selection time (the validation
set; the training-cohort hospitals) — but it demonstrates that the selection instability disclosed
elsewhere in the paper is addressable by a simple, no-new-experiments-required change to the selection
procedure: ranking candidates by a composite of patient-level and hospital-level performance rather than
by validation AUROC alone.